In [49]:
import warnings
warnings.filterwarnings("ignore")

In [50]:
from tqdm import tqdm
import pandas as pd
from datetime import datetime, timedelta
import sqlite3
import mysql.connector
import pyarrow

import numpy as np


In [29]:
connection = mysql.connector.connect(
    user = 'root',
    password = 'root',
    host = 'localhost',
    port = 3306,
    database = 'Historical_Data'
)
print("MySQL DB Connected")


MySQL DB Connected


In [30]:
cursor = connection.cursor()

cursor.execute("SELECT * FROM FT_HOUR_DATA WHERE Exchange = 'BTC-USD'")

results = cursor.fetchall()

columns = [column[0] for column in cursor.description]


df_original = pd.DataFrame(results, columns=columns)

In [31]:
df_original.sort_values(by = 'id_date', ascending = True)

,Open,High,Low,Close,adj_close,Volume,Hour,Exchange,id_exchange,id_date
0,29200,29302,29168,29268,29268,0,0,BTC-USD,193,20220521
17413,1962,1963,1944,1950,1950,0,1,ETH-USD,194,20220521
17414,1950,1961,1946,1959,1959,0,2,ETH-USD,194,20220521
17415,1958,1964,1950,1964,1964,0,3,ETH-USD,194,20220521
17416,1965,1965,1962,1963,1963,0,4,ETH-USD,194,20220521
...,...,...,...,...,...,...,...,...,...,...
17407,66815,66947,66785,66940,66940,0,19,BTC-USD,193,20240518
17408,66907,67033,66901,67006,67006,0,20,BTC-USD,193,20240518
17409,66992,67000,66852,66913,66913,0,21,BTC-USD,193,20240518
17399,67237,67372,67149,67227,67227,106489856,11,BTC-USD,193,20240518


In [32]:
df = df_original[['id_date', 'Hour', 'Close','Exchange']]

df['id_date'] = pd.to_datetime(df['id_date'], format='%Y%m%d')
df['datetime'] = df['id_date'] + pd.to_timedelta(df['Hour'], unit='h')

df = df[['datetime', 'Close', 'Exchange']]

In [33]:
df = df[df['Exchange'] == 'BTC-USD']

In [34]:
def get_cutoff_indices(
    data: pd.DataFrame,
    n_features: int, 
    step_size:int
) -> list:
    
    stop_position = len(data) - 1
    
    subseq_first_idex = 0
    subseq_mid_idx = n_features
    subseq_last_idx = n_features + 1
    indices = []
    
    while subseq_last_idx <= stop_position:
        indices.append((subseq_first_idex, subseq_mid_idx, subseq_last_idx))
        
        subseq_first_idex += step_size
        subseq_mid_idx += step_size
        subseq_last_idx += step_size
        
    return indices

In [35]:
from tqdm import tqdm

def transform_ts_data_into_features_and_target(
    ts_data: pd.DataFrame,
    input_seq_len: int,
    step_size: int
) -> pd.DataFrame:
    """
    Slices and transposes data from time-series format into a (features, target)
    format that we can use to train Supervised ML models
    """
    assert set(ts_data.columns) == {'datetime', 'Close', 'Exchange'}

    exchanges = ts_data['Exchange'].unique()
    #print(exchanges)
    features = pd.DataFrame()
    targets = pd.DataFrame()
    
    for exchange in tqdm(exchanges):
        
        # keep only ts data for this `location_id`
        ts_data_one_exchange = ts_data.loc[
            ts_data.Exchange == exchange, 
            ['datetime', 'Close']
        ]

        # pre-compute cutoff indices to split dataframe rows
        indices = get_cutoff_indices(
            ts_data_one_exchange,
            input_seq_len,
            step_size
        )

        # slice and transpose data into numpy arrays for features and targets
        n_examples = len(indices)
        x = np.ndarray(shape=(n_examples, input_seq_len), dtype=np.float32)
        y = np.ndarray(shape=(n_examples), dtype=np.float32)
        hours = []
        
        for i, idx in enumerate(indices):
            x[i, :] = ts_data_one_exchange.iloc[idx[0]:idx[1]]['Close'].values
            y[i] = ts_data_one_exchange.iloc[idx[1]:idx[2]]['Close'].values
            hours.append(ts_data_one_exchange.iloc[idx[1]]['datetime'])


        # numpy -> pandas
        features_one_exchange = pd.DataFrame(
            x,
            columns=[f'close_previous_{i+1}_hour' for i in reversed(range(input_seq_len))]
        )
        features_one_exchange['datetime'] = hours
        features_one_exchange['exchange'] = exchange

        # numpy -> pandas
        targets_one_exchange = pd.DataFrame(y, columns=[f'target_close_next_hour'])

        # concatenate results
        features = pd.concat([features, features_one_exchange])
        targets = pd.concat([targets, targets_one_exchange])

    features.reset_index(inplace=True, drop=True)
    targets.reset_index(inplace=True, drop=True)

    return features, targets['target_close_next_hour']

In [36]:
features, targets = transform_ts_data_into_features_and_target(
    df,
    input_seq_len=24, # one week of history -> 24*7*1
    step_size=24,
)

print(f'{features.shape=}')
print(f'{targets.shape=}')

100%|██████████| 1/1 [00:00<00:00,  1.68it/s]

features.shape=(725, 26)
targets.shape=(725,)


In [37]:
df = pd.concat([features, targets],
               axis = 1)

In [38]:
from typing import Tuple

def train_test_split(
    df: pd.DataFrame,
    cutoff_date: datetime,
    target_column_name: str,
    ) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    """
    """
    train_data = df[df.datetime < cutoff_date].reset_index(drop=True)
    test_data = df[df.datetime >= cutoff_date].reset_index(drop=True)

    X_train = train_data.drop(columns=[target_column_name])
    y_train = train_data[target_column_name]
    X_test = test_data.drop(columns=[target_column_name])
    y_test = test_data[target_column_name]

    return X_train, y_train, X_test, y_test

In [39]:
from datetime import datetime

X_train, y_train, X_test, y_test = train_test_split(
    df,
    cutoff_date=datetime(2024, 4, 1, 0, 0, 0),
    target_column_name='target_close_next_hour'
)

print(f'{X_train.shape=}')
print(f'{y_train.shape=}')
print(f'{X_test.shape=}')
print(f'{y_test.shape=}')

X_train.shape=(677, 26)
y_train.shape=(677,)
X_test.shape=(48, 26)
y_test.shape=(48,)


In [40]:
import xgboost as xgb

In [41]:
# use only past close data
past_close_columns = [c for c in X_train.columns if c.startswith('close_')]
X_train_only_numeric = X_train[past_close_columns]

#### XGBOOST

In [42]:
model = xgb.XGBRegressor()

In [43]:
model.fit(X_train_only_numeric, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=None, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=None, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=None, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [44]:
X_test_only_numeric = X_test[past_close_columns]
predictions = model.predict(X_test_only_numeric)
predictions

array([69848.445, 68031.164, 65598.164, 65598.164, 67538.266, 67556.01 ,
       69919.6  , 70597.34 , 71063.04 , 69761.46 , 70876.44 , 70928.1  ,
       66666.76 , 63664.38 , 65306.207, 63305.156, 62900.266, 60206.074,
       65041.316, 63298.664, 65620.48 , 65581.77 , 67127.85 , 67106.68 ,
       63225.805, 63220.285, 63011.62 , 62979.617, 63011.62 , 62353.082,
       59430.297, 59430.297, 59430.297, 61140.84 , 63011.62 , 63267.133,
       63011.62 , 63387.81 , 62687.773, 62881.58 , 60169.84 , 59430.297,
       61008.137, 62387.055, 60149.605, 65679.74 , 65619.086, 67118.01 ],
      dtype=float32)

In [45]:
from sklearn.metrics import mean_absolute_error
test_mae = mean_absolute_error(y_test, predictions)
print(f'{test_mae=:.4f}')

test_mae=879.5190


#### LIGHTGBM

In [46]:
import lightgbm as lgb

In [47]:
model_lgb = lgb.LGBMRegressor()
model_lgb.fit(X_train_only_numeric, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000317 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5424
[LightGBM] [Info] Number of data points in the train set: 677, number of used features: 24
[LightGBM] [Info] Start training from score 29476.571640
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

LGBMRegressor()

In [48]:
X_test_only_numeric = X_test[past_close_columns]
predictions = model_lgb.predict(X_test_only_numeric)

from sklearn.metrics import mean_absolute_error
test_mae = mean_absolute_error(y_test, predictions)
print(f'{test_mae=:.4f}')

test_mae=1137.4193
